# Custom CorrDiff Generation with Your Trained Checkpoints

This notebook demonstrates how to run CorrDiff inference using your own trained checkpoints from PhysicsNeMo.
We'll adapt the earth2studio CorrDiff framework to load your custom models and run generation on your data.

## Overview

- Load your custom regression and diffusion checkpoints
- Create a custom CorrDiff class for your models
- Set up data sources and coordinate systems for your domain
- Run inference and visualize results

## Prerequisites

- Your trained checkpoints accessible at `/app/host/home/younes.abid/git/physicsnemo/outputs/checkpoints/`
- Understanding of your model's input/output variables and domain
- Custom dataset configuration (similar to your `custom_list_2.py`)

## Environment Setup and Imports

In [14]:
#! pip install numba

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 36.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 14.2 MB/s  0:00:03m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [numba]32m1/2 [numba]

[notice] A new release of pip is available: 25.2 -> 26.0
[notice] To update, run: python3 -m pip install --upgrade pip


In [1]:
import os
import sys
import torch
import numpy as np
from pathlib import Path
from collections import OrderedDict
from datetime import datetime
import matplotlib.pyplot as plt
import cartopy.crs as ccrs

# Add the tutorials directory to path for our custom data loader
tutorials_path = '/app/host/home/younes.abid/git/earth2studio/notebooks/tutorials'
if tutorials_path not in sys.path:
    sys.path.append(tutorials_path)

# Earth2Studio imports
from earth2studio.data import DataSource, prep_data_array
from earth2studio.io import IOBackend, ZarrBackend
from earth2studio.utils.coords import map_coords, split_coords
from earth2studio.utils.time import to_time_array
from earth2studio.models.batch import batch_coords, batch_func
from earth2studio.models.dx.base import DiagnosticModel
from earth2studio.utils import handshake_coords, handshake_dim

# PhysicsNeMo imports
from physicsnemo.models import Module as PhysicsNemoModule
from physicsnemo.utils.generative import StackedRandomGenerator
from physicsnemo.utils.generative import deterministic_sampler as ablation_sampler

# Import our custom data loader
from custom_data_loader import CustomCorrDiffDataSource, create_fog_index_data_source

from loguru import logger

print('✅ All imports successful')

✅ All imports successful


## Check Your Checkpoint and Data Paths

In [2]:
! ls /app/host/mnt/storage/younes.abid/physicsnemo/data/custom_data_2/stats_432/stat.json

/app/host/mnt/storage/younes.abid/physicsnemo/data/custom_data_2/stats_432/stat.json


In [3]:
# Define paths to your trained checkpoints
base_path = '/app/host/home/younes.abid/git/physicsnemo/outputs/checkpoints/Fog_index'
regression_checkpoint = f'{base_path}/checkpoints_regression/UNet.0.390000.mdlus'

# You might also have diffusion checkpoints - adjust path as needed
diffusion_checkpoint = f'{base_path}/checkpoints_diffusion/EDMPrecondSuperResolution.0.590000.mdlus'  # Adjust if exists

# Data paths (adjust to your actual data location)
data_base_path = '/app/host/mnt/storage/younes.abid//physicsnemo/data/custom_data_2/ERA5_WRF_combined_concatenated_432'

stats_path = f'/app/host/mnt/storage/younes.abid/physicsnemo/data/custom_data_2/stats_432/stat.json'  # Adjust to actual stats file

print('Checking paths:')
print(f'Regression checkpoint: {regression_checkpoint}')
print(f'Exists: {os.path.exists(regression_checkpoint)}')

if os.path.exists(diffusion_checkpoint):
    print(f'Diffusion checkpoint: {diffusion_checkpoint}')
    print(f'Exists: {os.path.exists(diffusion_checkpoint)}')
else:
    print('Diffusion checkpoint not found - will use regression only mode')

print(f'Data base path: {data_base_path}')
print(f'Exists: {os.path.exists(data_base_path)}')

print(f'Stats path: {stats_path}')
print(f'Exists: {os.path.exists(stats_path)}')

# List available checkpoints
if os.path.exists(base_path):
    print(f'\nAvailable checkpoints in {base_path}:')
    for root, dirs, files in os.walk(base_path):
        for file in files:
            if file.endswith('.mdlus'):
                print(f'  {os.path.join(root, file)}')

# List available data files
if os.path.exists(data_base_path):
    print(f'\nAvailable data files in {data_base_path}:')
    for root, dirs, files in os.walk(data_base_path):
        for file in files:
            if file.endswith(('.nc', '.zarr')):
                print(f'  {os.path.join(root, file)}')

Checking paths:
Regression checkpoint: /app/host/home/younes.abid/git/physicsnemo/outputs/checkpoints/Fog_index/checkpoints_regression/UNet.0.390000.mdlus
Exists: True
Diffusion checkpoint: /app/host/home/younes.abid/git/physicsnemo/outputs/checkpoints/Fog_index/checkpoints_diffusion/EDMPrecondSuperResolution.0.590000.mdlus
Exists: True
Data base path: /app/host/mnt/storage/younes.abid//physicsnemo/data/custom_data_2/ERA5_WRF_combined_concatenated_432
Exists: True
Stats path: /app/host/mnt/storage/younes.abid/physicsnemo/data/custom_data_2/stats_432/stat.json
Exists: True

Available checkpoints in /app/host/home/younes.abid/git/physicsnemo/outputs/checkpoints/Fog_index:
  /app/host/home/younes.abid/git/physicsnemo/outputs/checkpoints/Fog_index/checkpoints_regression/UNet.0.355008.mdlus
  /app/host/home/younes.abid/git/physicsnemo/outputs/checkpoints/Fog_index/checkpoints_regression/UNet.0.370000.mdlus
  /app/host/home/younes.abid/git/physicsnemo/outputs/checkpoints/Fog_index/checkpoint

## Create Custom Data Source

We'll use our custom data loader that adapts your PhysicsNeMo data format to Earth2Studio.

In [4]:
# Configuration for your Fog_index experiment
# Adjust these paths and variables based on your actual setup

# Your actual data file paths (replace with your real file paths)
data_file_paths = [
    # Add your actual NetCDF file paths here
    "/app/host/mnt/storage/younes.abid//physicsnemo/data/custom_data_2/ERA5_WRF_combined_concatenated_432/2024-04-30_2024-05-30_21.nc"
]

# Check if you have actual data files - if not, we'll create a mock data source
if not data_file_paths or not all(os.path.exists(f) for f in data_file_paths):
    print('⚠️  No actual data files found. Creating mock data source for testing.')
    print('   Please update data_file_paths with your actual NetCDF files.')
    use_mock_data = True
else:
    use_mock_data = False

# Input variables (adjust to match your training)
input_variables =  ["t_850", "t_500", "z_850", "z_500", "u_850", "u_500", "v_850", "v_500", "u10", "v10", "t2m", "d2m", "skt", "sp", "tcwv", "tp"]

# Output variables for your Fog_index model
output_variables = [
    'Fog_index',  # Your primary output
]

print(f'Input variables ({len(input_variables)}): {input_variables}')
print(f'Output variables ({len(output_variables)}): {output_variables}')

Input variables (16): ['t_850', 't_500', 'z_850', 'z_500', 'u_850', 'u_500', 'v_850', 'v_500', 'u10', 'v10', 't2m', 'd2m', 'skt', 'sp', 'tcwv', 'tp']
Output variables (1): ['Fog_index']


In [5]:
if use_mock_data:
    # Create a mock data source that simulates your data structure
    from earth2studio.data import GFS  # Use GFS as fallback for testing
    
    class MockDataSource(DataSource):
        """Mock data source for testing when actual data is not available"""
        
        def __init__(self, input_vars):
            self.input_vars = input_vars
            # Use GFS as base for coordinate system
            self.gfs = GFS()
        
        def __call__(self, time, variable):
            # Filter to only requested variables that exist in GFS
            gfs_vars = ['tcwv', 'z500', 't500', 'u500', 'v500', 'z850', 't850', 'u850', 'v850', 't2m', 'u10m', 'v10m']
            available_vars = [v for v in variable if v in gfs_vars]
            
            if available_vars:
                return self.gfs(time, available_vars)
            else:
                # Create dummy data if no GFS variables requested
                import xarray as xr
                time_array = to_time_array(time)
                
                # Create dummy coordinate grids (adjust to your domain)
                lat = np.linspace(19.25, 28, 36, endpoint=True)
                lon = np.linspace(116, 126, 40, endpoint=False)
                
                data_vars = {}
                for var in variable:
                    data_vars[var] = (
                        ['time', 'lat', 'lon'],
                        np.random.randn(len(time_array), len(lat), len(lon)),
                        {'long_name': var, 'units': 'unknown'}
                    )
                
                coords = {
                    'time': time_array,
                    'lat': lat,
                    'lon': lon
                }
                
                return xr.Dataset(data_vars, coords=coords)
    
    # Use mock data source
    data_source = MockDataSource(input_variables)
    print('✅ Created mock data source for testing')
    
else:
    # Use your actual custom data source
    try:
        data_source = CustomCorrDiffDataSource(
            data_paths=data_file_paths,
            stats_path=stats_path,
            input_variables=input_variables,
            output_variables=output_variables,
            cache_data=True
        )
        print('✅ Created custom data source from your files')
    except Exception as e:
        print(f'❌ Error creating custom data source: {e}')
        print('   Falling back to mock data source')
        data_source = MockDataSource(input_variables)
        use_mock_data = True

🔍 Analyzing data files...
  File 1: 2024-04-30_2024-05-30_21.nc - 504 samples
✅ Total samples across all files: 504
📐 Input grid shape: (432, 432)
📐 Lat range: [19.25, 28.00]
📐 Lon range: [116.00, 125.98]
✅ Loaded normalization stats for 16 input variables
💾 Preloading data into memory...
  Loading file 1/1: 2024-04-30_2024-05-30_21.nc
    ✅ Loaded in 6.20s
✅ Data preloading complete!
✅ Created custom data source from your files


## Create Custom CorrDiff Class for Your Checkpoints

In [6]:
class CustomFogIndexCorrDiff(torch.nn.Module, DiagnosticModel):
    """
    Custom CorrDiff model for your Fog_index trained checkpoints.
    Adapted from earth2studio.models.dx.CorrDiffTaiwan for your specific domain and variables.
    """
    
    def __init__(
        self,
        regression_model: torch.nn.Module,
        residual_model: torch.nn.Module = None,
        input_variables: list = None,
        output_variables: list = None,
        input_grid: dict = None,
        output_grid: dict = None,
        normalization: dict = None,
        number_of_samples: int = 1,
        number_of_steps: int = 8,
        solver: str = 'euler'
    ):
        super().__init__()
        
        self.regression_model = regression_model
        self.residual_model = residual_model  # Can be None for regression-only
        
        # Define your input/output variables based on your Fog_index training
        self.input_vars = input_variables or input_variables
        self.output_vars = output_variables or output_variables
        
        # Define your coordinate grids - adjust to your actual domain
        self.input_grid = input_grid or {
            'lat': np.linspace(19.25, 28, 36, endpoint=True),  # Adjust to your domain
            'lon': np.linspace(116, 126, 40, endpoint=False),
        }
        
        # High resolution output grid (adjust based on your model)
        self.output_grid = output_grid or {
            'lat': np.linspace(19.0, 28.0, 448, endpoint=True),  # Your high-res grid
            'lon': np.linspace(116.0, 126.0, 448, endpoint=False),
        }
        
        # Normalization parameters - extract from your training stats
        self.normalization = normalization or {
            'in_center': torch.zeros(len(self.input_vars), 1, 1),
            'in_scale': torch.ones(len(self.input_vars), 1, 1),
            'out_center': torch.zeros(len(self.output_vars), 1, 1),
            'out_scale': torch.ones(len(self.output_vars), 1, 1)
        }
        
        # Register normalization parameters as buffers
        for key, value in self.normalization.items():
            self.register_buffer(key, value)
        
        self.number_of_samples = number_of_samples
        self.number_of_steps = number_of_steps
        self.solver = solver
    
    def input_coords(self):
        """Input coordinate system"""
        return OrderedDict({
            'batch': np.empty(0),
            'variable': np.array(self.input_vars),
            'lat': self.input_grid['lat'],
            'lon': self.input_grid['lon'],
        })
    
    @batch_coords()
    def output_coords(self, input_coords):
        """Output coordinate system"""
        
        output_coords = OrderedDict({
            'batch': np.empty(0),
            'sample': np.arange(self.number_of_samples),
            'variable': np.array(self.output_vars),
            'lat': self.output_grid['lat'],
            'lon': self.output_grid['lon'],
        })
        
        # Validate input coordinates
        target_input_coords = self.input_coords()
        handshake_dim(input_coords, 'lon', 3)
        handshake_dim(input_coords, 'lat', 2)
        handshake_dim(input_coords, 'variable', 1)
        handshake_coords(input_coords, target_input_coords, 'lon')
        handshake_coords(input_coords, target_input_coords, 'lat')
        handshake_coords(input_coords, target_input_coords, 'variable')
        
        output_coords['batch'] = input_coords['batch']
        return output_coords
    
    def _interpolate(self, x: torch.Tensor) -> torch.Tensor:
        """Interpolate from input grid to output grid"""
        # Simple implementation - you may need more sophisticated interpolation
        # For now, assume same grid or use basic interpolation
        if x.shape[-2:] == (len(self.output_grid['lat']), len(self.output_grid['lon'])):
            return x
        else:
            # Simple bilinear interpolation using torch.nn.functional.interpolate
            import torch.nn.functional as F
            target_size = (len(self.output_grid['lat']), len(self.output_grid['lon']))
            return F.interpolate(x, size=target_size, mode='bilinear', align_corners=True)
    
    @torch.inference_mode()
    def _forward(self, x: torch.Tensor) -> torch.Tensor:
        """Forward pass for a single batch"""
        
        # Interpolate to high-res grid if needed
        x = self._interpolate(x)
        
        # Normalize input
        x = (x - self.in_center) / self.in_scale
        
        # Add sample dimension
        x = x.unsqueeze(0)
        
        # Repeat for number of samples
        x = x.repeat(self.number_of_samples, 1, 1, 1)
        
        if self.residual_model is not None:
            # Full diffusion mode with residual model
            sample_seeds = torch.arange(self.number_of_samples)
            rnd = StackedRandomGenerator(x.device, sample_seeds)
            
            # Create latents
            img_resolution_x = len(self.output_grid['lat'])
            img_resolution_y = len(self.output_grid['lon'])
            latents = rnd.randn([
                self.number_of_samples,
                len(self.output_vars),
                img_resolution_x,
                img_resolution_y
            ], device=x.device)
            
            # Regression mean
            mean = self.unet_regression(
                self.regression_model,
                torch.zeros_like(latents),
                x,
                num_steps=self.number_of_steps
            )
            
            # Residual sampling
            res = ablation_sampler(
                self.residual_model,
                latents,
                x,
                randn_like=rnd.randn_like,
                num_steps=self.number_of_steps,
                solver=self.solver
            )
            
            x = mean + res
        else:
            # Regression-only mode
            # Create dummy latents for regression model
            img_resolution_x = len(self.output_grid['lat'])
            img_resolution_y = len(self.output_grid['lon'])
            latents = torch.zeros([
                self.number_of_samples,
                len(self.output_vars),
                img_resolution_x,
                img_resolution_y
            ], device=x.device)
            
            x = self.unet_regression(
                self.regression_model,
                latents,
                x,
                num_steps=self.number_of_steps
            )
            
        # Denormalize output
        x = self.out_scale * x + self.out_center
        
        return x
    
    @batch_func()
    def __call__(self, x: torch.Tensor, coords):
        """Forward pass of diagnostic model"""
        output_coords = self.output_coords(coords)
        
        out = torch.zeros(
            [len(v) for v in output_coords.values()],
            device=x.device,
            dtype=torch.float32
        )
        
        for i in range(out.shape[0]):
            out[i] = self._forward(x[i])
            
        return out, output_coords
    
    @staticmethod
    def unet_regression(net, latents, img_lr, num_steps=8, **kwargs):
        """U-Net regression with temporal sampling"""
        # Simplified version - adjust based on your model's requirements
        # Your regression model might expect different input format
        try:
            # Try the standard CorrDiff interface
            return net(latents, img_lr)
        except Exception as e:
            print(f'Standard interface failed: {e}')
            # Fallback to simpler interface
            try:
                return net(img_lr)
            except Exception as e2:
                print(f'Fallback interface also failed: {e2}')
                # Return dummy output
                return latents

print('✅ CustomFogIndexCorrDiff class defined')

✅ CustomFogIndexCorrDiff class defined


## Load Your Trained Models

Now let's load your actual trained checkpoints.

In [7]:
def load_normalization_from_data_source(data_source, input_vars, output_vars):
    """Extract normalization parameters from data source if available"""
    normalization = {
        'in_center': torch.zeros(len(input_vars), 1, 1),
        'in_scale': torch.ones(len(input_vars), 1, 1),
        'out_center': torch.zeros(len(output_vars), 1, 1),
        'out_scale': torch.ones(len(output_vars), 1, 1)
    }
    
    # Try to extract from custom data source
    if hasattr(data_source, 'input_mean') and hasattr(data_source, 'input_std'):
        normalization['in_center'] = torch.from_numpy(data_source.input_mean)
        normalization['in_scale'] = torch.from_numpy(data_source.input_std)
        print('✅ Extracted input normalization from data source')
    
    if hasattr(data_source, 'output_mean') and hasattr(data_source, 'output_std'):
        normalization['out_center'] = torch.from_numpy(data_source.output_mean)
        normalization['out_scale'] = torch.from_numpy(data_source.output_std)
        print('✅ Extracted output normalization from data source')
    
    return normalization


def load_custom_fog_index_model(regression_path, diffusion_path=None, data_source=None):
    """Load your custom CorrDiff model from checkpoints"""
    
    print(f'Loading regression model from: {regression_path}')
    try:
        regression_model = PhysicsNemoModule.from_checkpoint(regression_path).eval()
        print('✅ Regression model loaded successfully')
    except Exception as e:
        print(f'❌ Error loading regression model: {e}')
        return None
    
    residual_model = None
    if diffusion_path and os.path.exists(diffusion_path):
        print(f'Loading diffusion model from: {diffusion_path}')
        try:
            residual_model = PhysicsNemoModule.from_checkpoint(diffusion_path).eval()
            print('✅ Diffusion model loaded successfully')
        except Exception as e:
            print(f'⚠️  Error loading diffusion model: {e}')
            print('   Continuing with regression-only mode')
    else:
        print('⚠️  No diffusion model found - using regression-only mode')
    
    # Get normalization from data source if available
    normalization = load_normalization_from_data_source(
        data_source, input_variables, output_variables
    )
    
    # Create custom model
    model = CustomFogIndexCorrDiff(
        regression_model=regression_model,
        residual_model=residual_model,
        input_variables=input_variables,
        output_variables=output_variables,
        normalization=normalization
    )
    
    return model

# Load your model
try:
    corrdiff = load_custom_fog_index_model(
        regression_path=regression_checkpoint,
        diffusion_path=diffusion_checkpoint if os.path.exists(diffusion_checkpoint) else None,
        data_source=data_source
    )
    if corrdiff:
        print('✅ Custom Fog_index CorrDiff model created successfully')
    else:
        print('❌ Failed to create model')
except Exception as e:
    print(f'❌ Error loading model: {e}')
    print('This is expected if you need to adjust paths or model configuration')
    corrdiff = None

Loading regression model from: /app/host/home/younes.abid/git/physicsnemo/outputs/checkpoints/Fog_index/checkpoints_regression/UNet.0.390000.mdlus


/usr/local/lib/python3.11/dist-packages/physicsnemo/models/module.py:460: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model_dict = torch.load(


✅ Regression model loaded successfully
Loading diffusion model from: /app/host/home/younes.abid/git/physicsnemo/outputs/checkpoints/Fog_index/checkpoints_diffusion/EDMPrecondSuperResolution.0.590000.mdlus
✅ Diffusion model loaded successfully
✅ Extracted input normalization from data source
✅ Extracted output normalization from data source
✅ Custom Fog_index CorrDiff model created successfully


## Create Inference Workflow

This follows the Earth2Studio pattern but adapted for your custom model.

In [8]:
def run_custom_fog_index_inference(
    time,
    corrdiff,
    data,
    io,
    number_of_samples: int = 1
):
    """Custom Fog_index CorrDiff inference workflow"""
    
    logger.info('Running custom Fog_index CorrDiff inference!')
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    logger.info(f'Inference device: {device}')
    
    corrdiff = corrdiff.to(device)
    corrdiff.number_of_samples = number_of_samples
    
    # Fetch data from data source and load onto device
    time = to_time_array(time)
    x, coords = prep_data_array(
        data(time, corrdiff.input_coords()['variable']), device=device
    )
    x, coords = map_coords(x, coords, corrdiff.input_coords())
    
    logger.success(f'Fetched data from {data.__class__.__name__}')
    print(f'Input data shape: {x.shape}')
    print(f'Input coordinates: {coords}')
    
    # Set up IO backend
    output_coords = corrdiff.output_coords(corrdiff.input_coords())
    total_coords = OrderedDict({
        'time': coords['time'],
        'sample': output_coords['sample'],
        'lat': output_coords['lat'],
        'lon': output_coords['lon'],
    })
    io.add_array(total_coords, output_coords['variable'])
    
    logger.info('Inference starting!')
    x, coords = corrdiff(x, coords)
    io.write(*split_coords(x, coords))
    
    logger.success('Inference complete')
    return io

print('✅ Custom workflow function defined')

✅ Custom workflow function defined


## Run Inference

Now let's test the inference pipeline with your custom model.

In [9]:
# Only run inference if model was loaded successfully
if corrdiff is not None:
    # Set up IO backend
    io = ZarrBackend()
    
    # Define the time for inference
    # Adjust this timestamp to match your data availability
    if use_mock_data:
        # Use a standard timestamp for GFS data
        inference_time = ['2023-10-04T18:00:00']
    else:
        # Use a timestamp from your actual data
        inference_time = ['2023-01-01T00:00:00']  # Adjust to your data
    
    print(f'Running inference for time: {inference_time}')
    
    # Run inference
    try:
        io = run_custom_fog_index_inference(
            time=inference_time,
            corrdiff=corrdiff,
            data=data_source,
            io=io,
            number_of_samples=1
        )
        print('✅ Inference completed successfully!')
        
        # Show output information
        print(f'Output variables: {list(io.keys())}')
        for var in io.keys():
            if hasattr(io[var], 'shape'):
                print(f'  {var}: shape {io[var].shape}')
        
    except Exception as e:
        print(f'❌ Inference failed: {e}')
        import traceback
        traceback.print_exc()
        print('\nThis error is expected until paths and configurations are properly set up.')
else:
    print('❌ Cannot run inference: model not loaded')
    print('Please check your checkpoint paths and try again')

Running inference for time: ['2023-01-01T00:00:00']
2026-02-02 13:25:59.361 | INFO     | __main__:run_custom_fog_index_inference:10 - Running custom Fog_index CorrDiff inference!
2026-02-02 13:25:59.363 | INFO     | __main__:run_custom_fog_index_inference:12 - Inference device: cuda
🔄 Loading data for 1 times and 16 variables
✅ Loaded DataArray with shape: (1, 16, 432, 432)
2026-02-02 13:26:00.221 | SUCCESS  | __main__:run_custom_fog_index_inference:24 - Fetched data from CustomCorrDiffDataSource
Input data shape: torch.Size([1, 16, 36, 40])
Input coordinates: OrderedDict([('time', array(['2023-01-01T00:00:00.000000000'], dtype='datetime64[ns]')), ('variable', array(['t_850', 't_500', 'z_850', 'z_500', 'u_850', 'u_500', 'v_850',
       'v_500', 'u10', 'v10', 't2m', 'd2m', 'skt', 'sp', 'tcwv', 'tp'],
      dtype='<U5')), ('lat', array([19.25, 19.5 , 19.75, 20.  , 20.25, 20.5 , 20.75, 21.  , 21.25,
       21.5 , 21.75, 22.  , 22.25, 22.5 , 22.75, 23.  , 23.25, 23.5 ,
       23.75, 24.  ,

Traceback (most recent call last):
  File "/tmp/ipykernel_3510/1831934514.py", line 19, in <module>
    io = run_custom_fog_index_inference(
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_3510/1144793963.py", line 39, in run_custom_fog_index_inference
    x, coords = corrdiff(x, coords)
                ^^^^^^^^^^^^^^^^^^^
  File "/app/earth2studio/models/batch.py", line 178, in _wrapper
    out, out_coords = func(model, x, flatten_coords)
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_3510/4116860458.py", line 188, in __call__
    out[i] = self._forward(x[i])
             ^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/torch/utils/_contextlib.py", line 116, in decorate_context
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_3510/4116860458.py", line 107, in _forward
    x = self._interpolate(x)
        ^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_3510/4116860458.py", line 100, in _

## Visualization

Visualize your Fog_index results.

In [ ]:
def visualize_fog_index_results(io, save_path='outputs/custom_fog_index_prediction.jpg'):
    """Visualize the Fog_index inference results"""
    
    # Create output directory
    os.makedirs('outputs', exist_ok=True)
    
    # Variables to plot
    plot_vars = []
    var_info = {}
    
    # Check what variables are available
    if 'fog_index' in io:
        plot_vars.append('fog_index')
        var_info['fog_index'] = {'cmap': 'Blues', 'label': 'Fog Index'}
    
    if 't2m' in io:
        plot_vars.append('t2m')
        var_info['t2m'] = {'cmap': 'RdBu_r', 'label': '2m Temperature [K]'}
    
    if 'u10m' in io and 'v10m' in io:
        plot_vars.append('wind_speed')
        var_info['wind_speed'] = {'cmap': 'Greens', 'label': '10m Wind Speed [m/s]'}
    elif 'u10m' in io:
        plot_vars.append('u10m')
        var_info['u10m'] = {'cmap': 'RdBu_r', 'label': 'U-wind [m/s]'}
    
    if not plot_vars:
        print('❌ No recognizable variables found for plotting')
        print(f'Available variables: {list(io.keys())}')
        return
    
    # Create projection
    projection = ccrs.PlateCarree()
    
    # Create subplots
    n_vars = len(plot_vars)
    fig = plt.figure(figsize=(5 * n_vars, 6))
    
    for i, var in enumerate(plot_vars):
        ax = fig.add_subplot(1, n_vars, i+1, projection=projection)
        
        # Get data to plot
        if var == 'wind_speed' and 'u10m' in io and 'v10m' in io:
            # Calculate wind speed
            data = np.sqrt(io['u10m'][0, 0]**2 + io['v10m'][0, 0]**2)
        else:
            data = io[var][0, 0]  # [time, sample, lat, lon]
        
        # Plot the data
        c = ax.pcolormesh(
            io['lon'],
            io['lat'], 
            data,
            transform=ccrs.PlateCarree(),
            cmap=var_info[var]['cmap'],
            shading='auto'
        )
        
        # Add colorbar
        plt.colorbar(c, ax=ax, shrink=0.6, label=var_info[var]['label'])
        
        # Add map features
        ax.coastlines()
        ax.gridlines()
        ax.set_title(f'{var.replace("_", " ").title()}')
    
    plt.suptitle('Custom Fog Index CorrDiff Results', fontsize=16, y=0.95)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    print(f'✅ Visualization saved to: {save_path}')
    plt.show()

# Try to visualize if inference was successful
if 'io' in locals() and io is not None:
    try:
        visualize_fog_index_results(io)
    except Exception as e:
        print(f'Visualization failed: {e}')
        print('This is expected if inference did not complete successfully')
        
        # Show available data info for debugging
        print('\nAvailable data for debugging:')
        for key in io.keys():
            try:
                shape = io[key].shape if hasattr(io[key], 'shape') else 'unknown'
                print(f'  {key}: {shape}')
            except:
                print(f'  {key}: error accessing')
else:
    print('⚠️  No inference results to visualize')

## Configuration Guide

To fully adapt this notebook for your specific Fog_index setup:

### 1. **Update File Paths**
```python
# Update these paths to match your actual file locations:
data_file_paths = [
    '/app/host/path/to/your/fog_data_file_1.nc',
    '/app/host/path/to/your/fog_data_file_2.nc',
]
stats_path = '/app/host/path/to/your/normalization_stats.json'
```

### 2. **Configure Variables**
```python
# Adjust to match your training configuration:
input_variables = [
    # Your input meteorological variables
    'tcwv', 'z500', 't500', # ... etc
]

output_variables = [
    'fog_index',  # Your primary Fog Index output
    # Any additional outputs from your model
]
```

### 3. **Update Coordinate Grids**
```python
# In the CustomFogIndexCorrDiff class:
input_grid = {
    'lat': np.linspace(your_min_lat, your_max_lat, your_lat_points),
    'lon': np.linspace(your_min_lon, your_max_lon, your_lon_points),
}

output_grid = {
    'lat': np.linspace(your_hr_min_lat, your_hr_max_lat, your_hr_lat_points),
    'lon': np.linspace(your_hr_min_lon, your_hr_max_lon, your_hr_lon_points),
}
```

### 4. **Model Interface Adjustment**
The `unet_regression` method may need adjustment based on your model's interface.
Check your PhysicsNeMo model's forward method signature.

### 5. **Time Coordinate Mapping**
Implement proper time coordinate matching in the data source's `_find_time_indices` method.

## Debug and Model Inspection

In [ ]:
# Inspect your loaded model and data source
print('🔍 System Information:')
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'CUDA device: {torch.cuda.get_device_name()}')
    print(f'GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f}GB')

print('\n🏗️ Model Information:')
if corrdiff is not None:
    print(f'Model type: {type(corrdiff).__name__}')
    print(f'Input variables: {corrdiff.input_vars}')
    print(f'Output variables: {corrdiff.output_vars}')
    print(f'Number of samples: {corrdiff.number_of_samples}')
    print(f'Number of steps: {corrdiff.number_of_steps}')
    
    # Check input/output coordinates
    try:
        input_coords = corrdiff.input_coords()
        print(f'Input coordinate system:')
        for coord, values in input_coords.items():
            if hasattr(values, 'shape'):
                print(f'  {coord}: shape {values.shape}')
            else:
                print(f'  {coord}: {type(values)}')
    except Exception as e:
        print(f'Error getting input coordinates: {e}')
    
    # Model parameter count
    if hasattr(corrdiff, 'regression_model'):
        reg_params = sum(p.numel() for p in corrdiff.regression_model.parameters())
        print(f'Regression model parameters: {reg_params:,}')
    
    if hasattr(corrdiff, 'residual_model') and corrdiff.residual_model:
        res_params = sum(p.numel() for p in corrdiff.residual_model.parameters())
        print(f'Residual model parameters: {res_params:,}')
else:
    print('Model not loaded')

print('\n💾 Data Source Information:')
print(f'Data source type: {type(data_source).__name__}')
print(f'Using mock data: {use_mock_data}')
if hasattr(data_source, 'input_variables'):
    print(f'Available variables: {data_source.input_variables}')

if hasattr(data_source, 'total_samples'):
    print(f'Total samples: {data_source.total_samples}')

# Check GPU memory usage
if torch.cuda.is_available():
    print(f'\n🔧 Current GPU Memory:')
    print(f'Allocated: {torch.cuda.memory_allocated() / 1e9:.2f}GB')
    print(f'Cached: {torch.cuda.memory_reserved() / 1e9:.2f}GB')

In [44]:
# Debug data loading to understand the tensor conversion issue
print('🔍 Debugging Data Loading Issue')
print('=' * 50)

# Test 1: Check what our data source returns
print('\n1. Testing data source call:')
test_time = ['2023-01-01T00:00:00']
test_vars = input_variables[:3]  # Test with first 3 variables only

try:
    raw_data = data_source(test_time, test_vars)
    print(f'✅ Data source call succeeded')
    print(f'   Type: {type(raw_data)}')
    print(f'   Variables: {list(raw_data.data_vars.keys())}')
    print(f'   Coords: {list(raw_data.coords.keys())}')
    print(f'   Dims: {raw_data.dims}')
    
    # Test 2: Examine individual variables
    print('\n2. Testing individual variable data:')
    for var in test_vars:
        if var in raw_data:
            var_data = raw_data[var]
            print(f'\n   Variable: {var}')
            print(f'     Type: {type(var_data)}')
            print(f'     Shape: {var_data.shape}')
            print(f'     Dtype: {var_data.dtype}')
            print(f'     Values type: {type(var_data.values)}')
            
            # Try to access .values
            try:
                values = var_data.values
                print(f'     Values shape: {values.shape if hasattr(values, "shape") else "No shape"}')
                print(f'     Values dtype: {values.dtype if hasattr(values, "dtype") else "No dtype"}')
                print(f'     Is numpy array: {isinstance(values, np.ndarray)}')
            except Exception as e:
                print(f'     ❌ Error accessing .values: {e}')
                
                # Try alternative access methods
                try:
                    computed = var_data.compute() if hasattr(var_data, 'compute') else var_data
                    print(f'     After .compute(): {type(computed.values)}')
                except Exception as e2:
                    print(f'     ❌ Error with .compute(): {e2}')
                
                try:
                    loaded = var_data.load() if hasattr(var_data, 'load') else var_data
                    print(f'     After .load(): {type(loaded.values)}')
                except Exception as e3:
                    print(f'     ❌ Error with .load(): {e3}')

except Exception as e:
    print(f'❌ Data source call failed: {e}')
    import traceback
    traceback.print_exc()

# Test 3: Check your actual NetCDF file structure
print('\n3. Testing direct NetCDF file access:')
if data_file_paths and os.path.exists(data_file_paths[0]):
    try:
        import xarray as xr
        print(f'   Examining file: {data_file_paths[0]}')
        
        # Check file groups
        with xr.open_dataset(data_file_paths[0]) as ds:
            print(f'   Root groups/variables: {list(ds.keys())}')
            print(f'   Root coords: {list(ds.coords.keys())}')
            print(f'   Root dims: {ds.dims}')
        
        # Check input group
        try:
            with xr.open_dataset(data_file_paths[0], group='input') as ds_input:
                print(f'   Input group variables: {list(ds_input.keys())}')
                print(f'   Input group dims: {ds_input.dims}')
                
                # Test loading one variable
                if input_variables[0] in ds_input:
                    test_var = ds_input[input_variables[0]]
                    print(f'   Test variable {input_variables[0]}:')
                    print(f'     Shape: {test_var.shape}')
                    print(f'     Dtype: {test_var.dtype}')
                    print(f'     Chunks: {test_var.chunks if hasattr(test_var, "chunks") else "No chunks"}')
        except Exception as e:
            print(f'   ❌ Error accessing input group: {e}')
        
        # Check output group
        try:
            with xr.open_dataset(data_file_paths[0], group='output') as ds_output:
                print(f'   Output group variables: {list(ds_output.keys())}')
                print(f'   Output group dims: {ds_output.dims}')
        except Exception as e:
            print(f'   ❌ Error accessing output group: {e}')
            
    except Exception as e:
        print(f'   ❌ Error reading NetCDF file: {e}')
else:
    print('   ⚠️  No actual data files to examine')

# Test 4: Test Earth2Studio data preparation manually
print('\n4. Testing Earth2Studio data preparation:')
if 'raw_data' in locals():
    try:
        from earth2studio.data.utils import prep_data_array
        from earth2studio.utils.time import to_time_array
        
        print('   Testing prep_data_array...')
        time_array = to_time_array(test_time)
        x, coords = prep_data_array(raw_data, device='cpu')
        print(f'   ✅ prep_data_array succeeded!')
        print(f'     Output shape: {x.shape}')
        print(f'     Output type: {type(x)}')
        print(f'     Coords: {coords}')
    except Exception as e:
        print(f'   ❌ prep_data_array failed: {e}')
        
        # Test manual tensor conversion
        print('   Testing manual tensor conversion...')
        for var in test_vars:
            if var in raw_data:
                try:
                    var_data = raw_data[var]
                    print(f'     Variable {var}:')
                    
                    # Try different conversion methods
                    methods = [
                        ('direct .values', lambda x: x.values),
                        ('numpy array', lambda x: np.array(x)),
                        ('compute then values', lambda x: x.compute().values if hasattr(x, 'compute') else x.values),
                        ('load then values', lambda x: x.load().values if hasattr(x, 'load') else x.values),
                    ]
                    
                    for method_name, method_func in methods:
                        try:
                            result = method_func(var_data)
                            print(f'       ✅ {method_name}: {type(result)} {getattr(result, "shape", "no shape")}')
                            
                            # Try tensor conversion
                            try:
                                tensor = torch.tensor(result)
                                print(f'         Tensor conversion: ✅ {tensor.shape} {tensor.dtype}')
                                break  # Success, stop trying other methods
                            except Exception as te:
                                print(f'         Tensor conversion: ❌ {te}')
                                
                        except Exception as me:
                            print(f'       ❌ {method_name}: {me}')
                            
                except Exception as ve:
                    print(f'     ❌ Variable {var} processing failed: {ve}')

print('\n' + '=' * 50)
print('🔍 Debug analysis complete!')

🔍 Debugging Data Loading Issue

1. Testing data source call:
🔄 Loading data for 1 times and 3 variables
✅ Loaded dataset with shape: FrozenMappingWarningOnValuesAccess({'time': 1, 'lat': 432, 'lon': 432})
✅ Data source call succeeded
   Type: <class 'xarray.core.dataset.Dataset'>
   Variables: ['t_850', 't_500', 'z_850']
   Coords: ['time', 'lat', 'lon']
   Dims: FrozenMappingWarningOnValuesAccess({'time': 1, 'lat': 432, 'lon': 432})

2. Testing individual variable data:

   Variable: t_850
     Type: <class 'xarray.core.dataarray.DataArray'>
     Shape: (1, 432, 432)
     Dtype: float32
     Values type: <class 'numpy.ndarray'>
     Values shape: (1, 432, 432)
     Values dtype: float32
     Is numpy array: True

   Variable: t_500
     Type: <class 'xarray.core.dataarray.DataArray'>
     Shape: (1, 432, 432)
     Dtype: float32
     Values type: <class 'numpy.ndarray'>
     Values shape: (1, 432, 432)
     Values dtype: float32
     Is numpy array: True

   Variable: z_850
     Type: